# High-dimensional gene-expression ML

PCA and clustering, cross-validated classification, and FDR-controlled differential expression on a wide (genes >> samples) dataset. This demo uses the offline synthetic generator, which has a known set of differential genes to recover; `scripts/analyze.py --csv data/leukemia.csv` runs the same pipeline on the real Golub leukemia dataset.

In [1]:
from hdgenomics import (
    synthetic_expression, standardize, pca_embedding, kmeans_clusters,
    cross_val_classification, differential_expression,
)
from hdgenomics.dimreduce import cluster_agreement

data = synthetic_expression(n_samples=80, n_genes=2000, n_informative=40, effect_size=1.5)
data.n_samples, data.n_genes

(80, 2000)

In [2]:
xs, _ = standardize(data.X)
emb, evr = pca_embedding(xs, 2)
clusters = kmeans_clusters(emb, 2)
print('PCA explained variance (PC1, PC2):', evr.round(3))
print('K-means vs true labels ARI:', round(cluster_agreement(data.y, clusters), 3))
print('Logistic regression CV accuracy:', cross_val_classification(xs, data.y))

PCA explained variance (PC1, PC2): [0.02  0.017]
K-means vs true labels ARI: 0.95
Logistic regression CV accuracy: {'accuracy_mean': 0.9875, 'accuracy_std': 0.024999999999999998, 'n_splits': 5}


In [3]:
de = differential_expression(data.X, data.y, data.gene_names)
sig = [i for i, n in enumerate(data.gene_names) if n.startswith('sig_')]
recovered = de.significant[sig].mean()
print('Total significant genes at FDR 0.05:', int(de.significant.sum()))
print('Fraction of planted signal genes recovered:', round(float(recovered), 3))

Total significant genes at FDR 0.05: 41
Fraction of planted signal genes recovered: 0.975
